In [1]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from collections import Counter

df = pd.read_csv('../data/train_features.csv')

# Tách features và label
X = df.drop(columns=['isFraud'])
y = df['isFraud']

print(f"Shape X: {X.shape}")
print(f"Phân phối TRƯỚC SMOTE: {Counter(y)}")
print(f"Tỷ lệ fraud: {y.mean()*100:.2f}%")

Shape X: (10000, 192)
Phân phối TRƯỚC SMOTE: Counter({0: 9735, 1: 265})
Tỷ lệ fraud: 2.65%


In [2]:
# Chỉ áp dụng SMOTE trên tập train — KHÔNG áp dụng trên test!
# Chia train/test theo thời gian trước
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Fraud trong train TRƯỚC SMOTE: {Counter(y_train)}")

# Áp dụng SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\nFraud trong train SAU SMOTE: {Counter(y_train_sm)}")
print(f"Tỷ lệ fraud sau SMOTE: {y_train_sm.mean()*100:.2f}%")

Train: (8000, 192), Test: (2000, 192)
Fraud trong train TRƯỚC SMOTE: Counter({0: 7812, 1: 188})

Fraud trong train SAU SMOTE: Counter({0: 7812, 1: 7812})
Tỷ lệ fraud sau SMOTE: 50.00%


In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Trước SMOTE
axes[0].bar(['Legit', 'Fraud'], 
            [Counter(y_train)[0], Counter(y_train)[1]], 
            color=['steelblue', 'coral'])
axes[0].set_title('Trước SMOTE')
axes[0].set_ylabel('Số lượng')

# Sau SMOTE
axes[1].bar(['Legit', 'Fraud'], 
            [Counter(y_train_sm)[0], Counter(y_train_sm)[1]], 
            color=['steelblue', 'coral'])
axes[1].set_title('Sau SMOTE')
axes[1].set_ylabel('Số lượng')

plt.tight_layout()
plt.savefig('../reports/smote_comparison.png', dpi=80, bbox_inches='tight')
plt.close()
print("Đã lưu reports/smote_comparison.png!")

Đã lưu reports/smote_comparison.png!


In [4]:
# Lưu train đã SMOTE
train_smote = pd.DataFrame(X_train_sm, columns=X.columns)
train_smote['isFraud'] = y_train_sm.values
train_smote.to_csv('../data/train_smote.csv', index=False)

# Lưu test (KHÔNG SMOTE)
test_data = pd.DataFrame(X_test, columns=X.columns)
test_data['isFraud'] = y_test.values
test_data.to_csv('../data/test_data.csv', index=False)

print(f"Train SMOTE: {train_smote.shape}")
print(f"Test: {test_data.shape}")
print("Saved!")

C:\Users\minh4\AppData\Local\Temp\ipykernel_17316\1031086294.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_smote['isFraud'] = y_train_sm.values


Train SMOTE: (15624, 193)
Test: (2000, 193)
Saved!
